<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network 徽标">
    </a>
</p>


<h1>实验：使用小批量梯度下降的逻辑回归 </h1> 


<h2>目标</h2>

<ul>
    <li>将数据表示为 Dataset 对象</li>
    <li>使用 PyTorch 创建逻辑回归模型</li>
    <li>设置准则以计算损失</li>
    <li>创建数据加载器并设置批量大小</li>
    <li>创建优化器以更新模型参数并设置学习率</li>
    <li>训练模型</li>
</ul> 


<h2>目录</h2>
<p>在本实验中，你将学习如何使用小批量梯度下降训练 PyTorch 逻辑回归模型。</p>

-  [加载数据](#Load-Data)
-   [创建模型和总损失函数（Cost）](#Create-the-Model-and-Total-Loss-Function-(Cost))
-   [使用数据加载器设置批量大小](#Setting-the-Batch-Size-using-a-Data-Loader)
-   [设置学习率](#Setting-the-Learning-Rate)
-   [通过小批量梯度下降训练模型](#Train-the-Model-via-Mini-Batch-Gradient-Descent)
-   [问题](#Question)


<p>预计所需时间：<strong>30 分钟</strong></p>

<hr>


<h2>准备</h2>


我们需要以下库：**安装可能需要一些时间，请耐心等待……**


In [ ]:
!pip3 install torch torchvision torchaudio
!pip install matplotlib

In [ ]:
# 导入本实验所需的库

# 使用数组来操作和存储数据
import numpy as np
# 用于绘制数据和损失曲线
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
# PyTorch 库
import torch
# 用于创建数据集并执行小批量处理
from torch.utils.data import Dataset, DataLoader
# PyTorch 神经网络模块
import torch.nn as nn

<code>plot_error_surfaces</code> 类仅用于帮助你在训练期间可视化数据空间和参数空间，与 Pytorch 本身无关。


In [ ]:
# 创建用于绘图的类和函数

class plot_error_surfaces(object):
    
    # 构造函数
    def __init__(self, w_range, b_range, X, Y, n_samples = 30, go = True):
        W = np.linspace(-w_range, w_range, n_samples)
        B = np.linspace(-b_range, b_range, n_samples)
        w, b = np.meshgrid(W, B)    
        Z = np.zeros((30, 30))
        count1 = 0
        self.y = Y.numpy()
        self.x = X.numpy()
        for w1, b1 in zip(w, b):
            count2 = 0
            for w2, b2 in zip(w1, b1):
                yhat= 1 / (1 + np.exp(-1*(w2*self.x+b2)))
                Z[count1,count2]=-1*np.mean(self.y*np.log(yhat+1e-16) +(1-self.y)*np.log(1-yhat+1e-16))
                count2 += 1   
            count1 += 1
        self.Z = Z
        self.w = w
        self.b = b
        self.W = []
        self.B = []
        self.LOSS = []
        self.n = 0
        if go == True:
            plt.figure()
            plt.figure(figsize=(7.5, 5))
            plt.axes(projection='3d').plot_surface(self.w, self.b, self.Z, rstride=1, cstride=1, cmap='viridis', edgecolor='none')
            plt.title('Loss Surface')
            plt.xlabel('w')
            plt.ylabel('b')
            plt.show()
            plt.figure()
            plt.title('Loss Surface Contour')
            plt.xlabel('w')
            plt.ylabel('b')
            plt.contour(self.w, self.b, self.Z)
            plt.show()
            
     # 设置器
    def set_para_loss(self, model, loss):
        self.n = self.n + 1
        self.W.append(list(model.parameters())[0].item())
        self.B.append(list(model.parameters())[1].item())
        self.LOSS.append(loss)
    
    # 绘制图表
    def final_plot(self): 
        ax = plt.axes(projection='3d')
        ax.plot_wireframe(self.w, self.b, self.Z)
        ax.scatter(self.W, self.B, self.LOSS, c='r', marker='x', s=200, alpha=1)
        plt.figure()
        plt.contour(self.w, self.b, self.Z)
        plt.scatter(self.W, self.B, c='r', marker='x')
        plt.xlabel('w')
        plt.ylabel('b')
        plt.show()
        
    # 绘制图表
    def plot_ps(self):
        plt.subplot(121)
        plt.ylim
        plt.plot(self.x[self.y==0], self.y[self.y==0], 'ro', label="training points")
        plt.plot(self.x[self.y==1], self.y[self.y==1]-1, 'o', label="training points")
        plt.plot(self.x, self.W[-1] * self.x + self.B[-1], label="estimated line")
        plt.xlabel('x')
        plt.ylabel('y')
        plt.ylim((-0.1, 2))
        plt.title('Data Space Iteration: ' + str(self.n))
        plt.show()
        plt.subplot(122)
        plt.contour(self.w, self.b, self.Z)
        plt.scatter(self.W, self.B, c='r', marker='x')
        plt.title('Loss Surface Contour Iteration' + str(self.n))
        plt.xlabel('w')
        plt.ylabel('b')
        
# 绘制图表

def PlotStuff(X, Y, model, epoch, leg=True):
    
    plt.plot(X.numpy(), model(X).detach().numpy(), label=('epoch ' + str(epoch)))
    plt.plot(X.numpy(), Y.numpy(), 'r')
    if leg == True:
        plt.legend()
    else:
        pass

设置随机种子：


In [ ]:
# 设置随机种子可以控制随机性并保证结果可复现
torch.manual_seed(0)

<!--Empty Space for separating topics-->


<h2 id="Makeup_Data">加载数据</h2>


Dataset 类代表一个数据集。你的自定义数据集应继承上面导入的 Dataset，并重写以下方法：

<p><code>__len__</code> 使得 len(dataset) 返回数据集的大小。</p>
<p><code>__getitem__</code> 支持索引，以便可以使用 dataset[i] 获取第 i 个样本</p>

下面我们将创建一个示例数据集


In [ ]:
# 创建继承 Dataset 的自定义 Data 类
class Data(Dataset):
    
    # 构造函数
    def __init__(self):
        # 创建从 -1 到 1、步长为 0.1 的 X 值
        self.x = torch.arange(-1, 1, 0.1).view(-1, 1)
        # 创建全部设为 0 的 Y 值
        self.y = torch.zeros(self.x.shape[0], 1)
        # 将大于 0.2 的 X 值对应的 Y 设为 1
        self.y[self.x[:, 0] > 0.2] = 1
        # 设置 .len 属性，因为我们需要重写 __len__ 方法
        self.len = self.x.shape[0]
    
    # 返回指定索引处数据的获取器
    def __getitem__(self, index):      
        return self.x[index], self.y[index]
    
    # 获取数据集长度
    def __len__(self):
        return self.len

创建 <code>Data</code> 对象


In [ ]:
# 创建 Data 对象
data_set = Data()

我们可以查看数据集的 X 值


In [ ]:
data_set.x

我们可以查看数据集的 Y 值，它们对应于 X 值的类别


In [ ]:
data_set.y

我们可以获取数据集的长度


In [ ]:
len(data_set)

我们可以获取第一个样本的标签 $y$ 和 $x$ 


In [ ]:
x,y = data_set[0]
print("x = {},  y = {}".format(x,y))

我们可以获取第二个样本的标签 $y$ 和 $x$：


In [ ]:
x,y = data_set[1]
print("x = {},  y = {}".format(x,y))

 我们可以看到可以将这个一维数据集分成两个类别：


In [ ]:
plt.plot(data_set.x[data_set.y==0], data_set.y[data_set.y==0], 'ro', label="y=0")
plt.plot(data_set.x[data_set.y==1], data_set.y[data_set.y==1]-1, 'o', label="y=1")
plt.xlabel('x')
plt.legend()          

<!--Empty Space for separating topics-->


<h2 id="Model_Cost">创建模型和总损失函数（Cost）</h2>


对于逻辑回归，通常我们不会使用 PyTorch，而是使用 Scikit-Learn，因为它更易于使用和设置。我们在这里使用 PyTorch 是因为这对深度学习来说是很好的练习。Scikit-Learn 通常用于机器学习，而 PyTorch 通常用于深度学习。


我们将创建一个自定义类，用于定义基于 PyTorch 的逻辑回归架构。逻辑回归有一个单层，输入是数据集中 X 值的特征数量（X 的维度），输出为单个值。该层的输出会传入 sigmoid 函数，sigmoid 是一个介于 0 和 1 之间的函数。层输出越大，越接近 1；输出越小，越接近 0。sigmoid 函数使我们能够将此输出转化为分类问题：输出值越接近 1 属于一个类别，越接近 0 属于另一个类别。


Sigmoid 函数

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-Coursera/labs/Module3/Logistic-curve.svg)


In [ ]:
# 创建继承 nn.Module 的 logistic_regression 类，nn.Module 是所有神经网络的基类
class logistic_regression(nn.Module):
    
    # 构造函数
    def __init__(self, n_inputs):
        super(logistic_regression, self).__init__()
        # 逻辑回归的单层，输入数量为 n_inputs，输出为 1
        self.linear = nn.Linear(n_inputs, 1)
        
    # 预测
    def forward(self, x):
        # 将输入 x 传入上面定义的单层，再将输出传入 sigmoid 函数并返回结果
        yhat = torch.sigmoid(self.linear(x))
        return yhat

我们可以检查 X 值具有的特征数量、输入大小或 X 的维度


In [ ]:
x,y = data_set[0]
len(x)

创建一个逻辑回归对象或模型，输入参数为维度数量。


In [ ]:
# 创建 logistic_regression 结果

model = logistic_regression(1)

我们可以进行预测 sigma $\sigma$，这使用上面定义的 forward 函数


In [ ]:
x = torch.tensor([-1.0])

sigma = model(x)
sigma

我们也可以使用我们的数据进行预测


In [ ]:
x,y = data_set[2]

sigma = model(x)
sigma

创建一个 <code>plot_error_surfaces</code> 对象，以便在训练期间可视化数据空间和可学习参数空间：

在损失曲面图上，我们可以看到损失值随 w 和 b 的变化而变化，黄色表示高损失，深蓝色表示低损失，这正是我们想要的

在损失曲面等高线图上，我们可以从俯视角度查看损失曲面图


In [ ]:
# 创建 plot_error_surfaces 对象

# 15 是 w 的取值范围
# 13 是 b 的取值范围
# data_set[:][0] 是所有 X 值
# data_set[:][1] 是所有 Y 值

get_surface = plot_error_surfaces(15, 13, data_set[:][0], data_set[:][1])

我们使用二元交叉熵损失定义准则。它将度量预测值与真实值之间的差异/损失。


In [ ]:
criterion = nn.BCELoss()

我们已有样本：


In [ ]:
x, y = data_set[0]
print("x = {},  y = {}".format(x,y))

我们可以使用模型进行预测：


In [ ]:
sigma = model(x)
sigma

我们可以计算损失 


In [ ]:
loss = criterion(sigma, y)
loss

## 使用数据加载器设置批量大小


你必须使用 PyTorch 中的 data loader，它会输出一批数据；输入是 <code>dataset</code> 和 <code>batch_size</code>


In [ ]:
batch_size=10

In [ ]:
trainloader = DataLoader(dataset = data_set, batch_size = 10)

In [ ]:
dataset_iter = iter(trainloader)

In [ ]:
X,y=next(dataset_iter )

我们可以看到这里的 10 个值与我们的批量大小相同


In [ ]:
X

## 设置学习率


我们可以通过将学习率设置为优化器中的参数来设置学习率，同时传入我们正在训练的逻辑回归模型的参数。优化器 torch.optim.SGD 的作用是使用准则生成的损失，根据学习率更新模型参数。SGD 代表随机梯度下降，通常意味着批量大小设置为 1，但我们在上面设置的数据加载器已将其转变为小批量梯度下降。


In [ ]:
learning_rate = 0.1

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

<!--Empty Space for separating topics-->


<h2 id="BGD">通过小批量梯度下降训练模型</h2>


我们将使用不同的批量大小和学习率来训练模型。


### 小批量梯度下降


在本例中，我们将数据加载器的批量大小设置为 5，并将 epoch 数量设置为 250。


首先，我们必须重新创建 get_surface 对象，以便每个示例只获得该模型的损失曲面。


In [ ]:
get_surface = plot_error_surfaces(15, 13, data_set[:][0], data_set[:][1], 30)

#### 训练模型


In [ ]:
# 首先创建我们要训练的模型实例
model = logistic_regression(1)
# 创建一个用于度量损失的准则
criterion = nn.BCELoss()
# 使用数据集创建数据加载器，并指定批量大小为 5
trainloader = DataLoader(dataset = data_set, batch_size = 5)
# 使用模型参数和学习率创建优化器
optimizer = torch.optim.SGD(model.parameters(), lr = .01)
# 然后设置 epoch 数量，即在整个训练数据集上训练的总轮数
epochs=500
# 这将保存每次迭代的损失，以便最后绘制
loss_values = []

# 循环将执行指定数量的 epoch
for epoch in range(epochs):
    # 对于训练数据中的每个批次
    for x, y in trainloader:
        # 根据 X 值进行预测
        yhat = model(x)
        # 度量预测值与真实 Y 值之间的损失
        loss = criterion(yhat, y)
        # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置就会累积
        optimizer.zero_grad()
        # 计算每个权重和偏置的梯度值
        loss.backward()
        # 根据计算得到的梯度值更新权重和偏置
        optimizer.step()
        # 为损失曲面等高线图设置参数
        get_surface.set_para_loss(model, loss.tolist())
        # 保存本次迭代的损失
        loss_values.append(loss)
    # 每 20 个 epoch 打印当前迭代的数据空间
    if epoch % 20 == 0:
        get_surface.plot_ps()

我们可以查看权重和偏置的最终值。该权重和偏置对应于数据空间图中的橙色线，以及损失曲面等高线图中 X 的最终位置。


In [ ]:
w = model.state_dict()['linear.weight'].data[0]
b = model.state_dict()['linear.bias'].data[0]
print("w = ", w, "b = ", b)

现在我们可以获得训练数据的准确率


In [ ]:
# 获取预测值
yhat = model(data_set.x)
# 将预测值四舍五入为 0 或 1 的整数以表示类别
yhat = torch.round(yhat)
# 用于记录正确预测数量的计数器
correct = 0
# 遍历每个预测值和实际 y 值
for prediction, actual in zip(yhat, data_set.y):
    # 比较预测值和实际 y 值是否相同
    if (prediction == actual):
        # 如果预测正确，则计数器加 1
        correct+=1
# 通过将正确预测数除以数据集长度来输出准确率
print("Accuracy: ", correct/len(data_set)*100, "%")

最后，我们绘制代价与迭代次数的关系图；虽然曲线有些波动，但整体呈下降趋势。


In [ ]:
LOSS_BGD1=[]
for i in loss_values:
    LOSS_BGD1.append(i.item())

plt.plot(LOSS_BGD1)
plt.xlabel("Iteration")
plt.ylabel("Cost")


### 随机梯度下降


在本例中，我们将数据加载器的批量大小设置为 1，以便对每个样本执行梯度下降，这被称为随机梯度下降。epoch 数量设置为 100。

注意，在本例中批量大小从 5 减少到 1，因此会有更多的迭代。因此，我们可以通过减少 epoch 数量来减少迭代次数。由于批量变小，我们优化得更频繁，所以不需要那么多的 epoch。

首先，我们必须重新创建 `get_surface` 对象，以便每个示例只获得该模型的损失曲面。


In [ ]:
get_surface = plot_error_surfaces(15, 13, data_set[:][0], data_set[:][1], 30)

#### 训练模型


In [ ]:
# 首先创建我们要训练的模型实例
model = logistic_regression(1)
# 创建一个用于度量损失的准则
criterion = nn.BCELoss()
# 使用数据集创建数据加载器，并指定批量大小为 1
trainloader = DataLoader(dataset = data_set, batch_size = 1)
# 使用模型参数和学习率创建优化器
optimizer = torch.optim.SGD(model.parameters(), lr = .01)
# 然后设置 epoch 数量，即在整个训练数据集上训练的总轮数
epochs=100
# 这将保存每次迭代的损失，以便最后绘制
loss_values = []

# 循环将执行指定数量的 epoch
for epoch in range(epochs):
    # 对于训练数据中的每个批次
    for x, y in trainloader:
        # 根据 X 值进行预测
        yhat = model(x)
        # 度量预测值与真实 Y 值之间的损失
        loss = criterion(yhat, y)
        # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置就会累积
        optimizer.zero_grad()
        # 计算每个权重和偏置的梯度值
        loss.backward()
        # 根据计算得到的梯度值更新权重和偏置
        optimizer.step()
        # 为损失曲面等高线图设置参数
        get_surface.set_para_loss(model, loss.tolist())
        # 保存本次迭代的损失
        loss_values.append(loss)
    # 每 20 个 epoch 打印当前迭代的数据空间
    if epoch % 20 == 0:
        get_surface.plot_ps()

我们可以查看权重和偏置的最终值。该权重和偏置对应于数据空间图中的橙色线，以及损失曲面等高线图中 X 的最终位置。


In [ ]:
w = model.state_dict()['linear.weight'].data[0]
b = model.state_dict()['linear.bias'].data[0]
print("w = ", w, "b = ", b)

现在我们可以获得训练数据的准确率


In [ ]:
# 获取预测值
yhat = model(data_set.x)
# 将预测值四舍五入为 0 或 1 的整数以表示类别
yhat = torch.round(yhat)
# 用于记录正确预测数量的计数器
correct = 0
# 遍历每个预测值和实际 y 值
for prediction, actual in zip(yhat, data_set.y):
    # 比较预测值和实际 y 值是否相同
    if (prediction == actual):
        # 如果预测正确，则计数器加 1
        correct+=1
# 通过将正确预测数除以数据集长度来输出准确率
print("Accuracy: ", correct/len(data_set)*100, "%")

最后，我们绘制代价与迭代次数的关系图；虽然曲线有些波动，但整体呈下降趋势。


In [ ]:
LOSS_BGD1=[]
for i in loss_values:
    LOSS_BGD1.append(i.item())

 
plt.plot(LOSS_BGD1)
plt.xlabel("Iteration")
plt.ylabel("Cost")


### 高学习率


在本例中，我们将数据加载器的批量大小设置为 1，以便对每个样本执行梯度下降，这被称为随机梯度下降。这次学习率将设置为 0.1，以代表较高的学习率，我们将观察训练时会发生什么。

首先，我们必须重新创建 `get_surface` 对象，以便每个示例只获得该模型的损失曲面。


In [ ]:
get_surface = plot_error_surfaces(15, 13, data_set[:][0], data_set[:][1], 30)

#### 训练模型


In [ ]:
# 首先创建我们要训练的模型实例
model = logistic_regression(1)
# 创建一个用于度量损失的准则
criterion = nn.BCELoss()
# 使用数据集创建数据加载器，并指定批量大小为 1
trainloader = DataLoader(dataset = data_set, batch_size = 1)
# 使用模型参数和学习率创建优化器
optimizer = torch.optim.SGD(model.parameters(), lr = 1)
# 然后设置 epoch 数量，即在整个训练数据集上训练的总轮数
epochs=100
# 这将保存每次迭代的损失，以便最后绘制
loss_values = []

# 循环将执行指定数量的 epoch
for epoch in range(epochs):
    # 对于训练数据中的每个批次
    for x, y in trainloader:
        # 根据 X 值进行预测
        yhat = model(x)
        # 度量预测值与真实 Y 值之间的损失
        loss = criterion(yhat, y)
        # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置就会累积
        optimizer.zero_grad()
        # 计算每个权重和偏置的梯度值
        loss.backward()
        # 根据计算得到的梯度值更新权重和偏置
        optimizer.step()
        # 为损失曲面等高线图设置参数
        get_surface.set_para_loss(model, loss.tolist())
        # 保存本次迭代的损失
        loss_values.append(loss)
    # 每 20 个 epoch 打印当前迭代的数据空间
    if epoch % 20 == 0:
        get_surface.plot_ps()

注意，在这个示例中，由于学习率较高，损失曲面等高线图比上一个示例有更大幅度的移动，并且由于超过了最小值，还会向多个方向移动。


我们可以查看权重和偏置的最终值。该权重和偏置对应于数据空间图中的橙色线，以及损失曲面等高线图中 X 的最终位置。


In [ ]:
w = model.state_dict()['linear.weight'].data[0]
b = model.state_dict()['linear.bias'].data[0]
print("w = ", w, "b = ", b)

现在我们可以获得训练数据的准确率


In [ ]:
# 获取预测值
yhat = model(data_set.x)
# 将预测值四舍五入为 0 或 1 的整数以表示类别
yhat = torch.round(yhat)
# 用于记录正确预测数量的计数器
correct = 0
# 遍历每个预测值和实际 y 值
for prediction, actual in zip(yhat, data_set.y):
    # 比较预测值和实际 y 值是否相同
    if (prediction == actual):
        # 如果预测正确，则计数器加 1
        correct+=1
# 通过将正确预测数除以数据集长度来输出准确率
print("Accuracy: ", correct/len(data_set)*100, "%")

最后，我们绘制代价与迭代次数的关系图；虽然曲线有些波动，但整体呈下降趋势。


In [ ]:
LOSS_BGD1=[]
for i in loss_values:
    LOSS_BGD1.append(i.item())

 
plt.plot(LOSS_BGD1)
plt.xlabel("Iteration")
plt.ylabel("Cost")


## 问题


使用以下代码训练模型，设置 `learning rate` 为 `.01`，`120 epochs`，`batch_size` 为 `1`。


In [ ]:
get_surface = plot_error_surfaces(15, 13, data_set[:][0], data_set[:][1], 30)

#### 训练模型


In [ ]:
# 首先创建我们要训练的模型实例
model = logistic_regression(1)
# 创建一个用于度量损失的准则
criterion = nn.BCELoss()
# 使用数据集创建数据加载器，并指定批量大小为 1
trainloader = DataLoader(dataset = data_set, batch_size = 1)
# 使用模型参数和学习率创建优化器
optimizer = torch.optim.SGD(model.parameters(), lr = .01)
# 然后设置 epoch 数量，即在整个训练数据集上训练的总轮数
epochs= 120
# 这将保存每次迭代的损失，以便最后绘制
loss_values = []
# 循环将执行指定数量的 epoch
for epoch in range(epochs):
    # 对于训练数据中的每个批次
    for x, y in trainloader:
        # 根据 X 值进行预测
        yhat = model(x)
        # 度量预测值与真实 Y 值之间的损失
        loss = criterion(yhat, y)
        # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置就会累积
        optimizer.zero_grad()
        # 计算每个权重和偏置的梯度值
        loss.backward()
        # 根据计算得到的梯度值更新权重和偏置
        optimizer.step()
        # 为损失曲面等高线图设置参数
        get_surface.set_para_loss(model, loss.tolist())
        # 保存本次迭代的损失
        loss_values.append(loss)
    # 每 20 个 epoch 打印当前迭代的数据空间
    if epoch % 20 == 0:
        get_surface.plot_ps()

<details><summary>点击查看答案</summary>
<code>  
# 首先创建我们要训练的模型实例
model = logistic_regression(1)
# 创建一个用于度量损失的准则
criterion = nn.BCELoss()
# 使用数据集创建数据加载器，并指定批量大小为 1
trainloader = DataLoader(dataset = data_set, batch_size = 1)
# 使用模型参数和学习率创建优化器
optimizer = torch.optim.SGD(model.parameters(), lr = .01)
# 然后设置 epoch 数量，即在整个训练数据集上训练的总轮数
epochs= 120
# 这将保存每次迭代的损失，以便最后绘制
loss_values = []
# 循环将执行指定数量的 epoch
for epoch in range(epochs):
    # 对于训练数据中的每个批次
    for x, y in trainloader:
        # 根据 X 值进行预测
        yhat = model(x)
        # 度量预测值与真实 Y 值之间的损失
        loss = criterion(yhat, y)
        # 重置计算得到的梯度值，每次都必须这样做，因为如果不重置就会累积
        optimizer.zero_grad()
        # 计算每个权重和偏置的梯度值
        loss.backward()
        # 根据计算得到的梯度值更新权重和偏置
        optimizer.step()
        # 为损失曲面等高线图设置参数
        get_surface.set_para_loss(model, loss.tolist())
        # 保存本次迭代的损失
        loss_values.append(loss)
    # 每 20 个 epoch 打印当前迭代的数据空间
    if epoch % 20 == 0:
        get_surface.plot_ps()
</code>
</details>


我们可以查看权重和偏置的最终值。该权重和偏置对应于数据空间图中的橙色线，以及损失曲面等高线图中 X 的最终位置。


In [ ]:
w = model.state_dict()['linear.weight'].data[0]
b = model.state_dict()['linear.bias'].data[0]
print("w = ", w, "b = ", b)

现在我们可以获得训练数据的准确率


In [ ]:
# 获取预测值
yhat = model(data_set.x)
# 将预测值四舍五入为 0 或 1 的整数以表示类别
yhat = torch.round(yhat)
# 用于记录正确预测数量的计数器
correct = 0
# 遍历每个预测值和实际 y 值
for prediction, actual in zip(yhat, data_set.y):
    # 比较预测值和实际 y 值是否相同
    if (prediction == actual):
        # 如果预测正确，则计数器加 1
        correct+=1
# 通过将正确预测数除以数据集长度来输出准确率
print("Accuracy: ", correct/len(data_set)*100, "%")

最后，我们绘制代价与迭代次数的关系图；虽然曲线有些波动，但整体呈下降趋势。


In [ ]:
LOSS_BGD1=[]
for i in loss_values:
    LOSS_BGD1.append(i.item())

 
plt.plot(LOSS_BGD1)
plt.xlabel("Iteration")
plt.ylabel("Cost")


<!--Empty Space for separating topics-->


<h2>关于作者：</h2> 

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> 拥有电气工程博士学位，其研究专注于使用机器学习、信号处理和计算机视觉来确定视频如何影响人类认知。Joseph 自完成博士学位以来一直在 IBM 工作。


其他贡献者：<a href="https://www.linkedin.com/in/michelleccarey/">Michelle Carey</a>、<a href="www.linkedin.com/in/jiahui-mavis-zhou-a4537814a">Mavis Zhou</a>


<!--## Change Log

| 日期（YYYY-MM-DD） | 版本 | 修改者 | 变更说明                                          |
| ----------------- | ------- | ---------- | ----------------------------------------------------------- |
| 2025-07-10        | 2.0     | Sathya    | 将实验转换为 Jupyter Current 版本 |
| 2020-09-23        | 2.0     | Shubham    | 将实验迁移到 Markdown 并添加到 GitLab 课程仓库中 |-->


<hr>


## <h3 align="center"> © IBM Corporation。保留所有权利。 <h3/>
